### Building test case generator with multi-system using auto-gen 


In [ ]:
%pip install autogen
%pip install "autogen[openai]"
%pip install "autogen-ext[ollama]"


In [29]:
from autogen import AssistantAgent, UserProxyAgent

llm_config = {
    "config_list": [
        {
            "model": "gpt-oss:20b-cloud",
            #"model_name": "gpt-oss",
            "temperature": 0.7,
            "base_url": "http://localhost:11434/v1",
            "api_key": "ollama"

        }
    ]
}

### Create an assistant agent 

In [30]:
qa_assistant = AssistantAgent(
    name="QAAssistant",
    description="An assistant that answers questions based on provided context.",
    system_message="You are a helpful QA assistant who can write BDD scenarios and you create not more than the number of test cases that the user asks for.",
    llm_config=llm_config
)

### Create a user proxy agent 

In [31]:


qa_lead = UserProxyAgent(
    name="QALead",
    description="A QA lead who assigns tasks to the QA assistant.",
    system_message="You are a QA lead who assigns tasks to the QA assistant.",
    code_execution_config=False,
    human_input_mode="NEVER"
)

### Initiate the chat 

In [32]:
qa_lead.initiate_chat(
    recipient=qa_assistant,
    message="Create 3 BDD test cases for a login feature that includes username and password validation.",
    max_turns=3
)


QALead (to QAAssistant):

Create 3 BDD test cases for a login feature that includes username and password validation.

--------------------------------------------------------------------------------
[autogen.oai.client: 12-21 16:06:55] {734} WARNING - Model gpt-oss:20b-cloud is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
QAAssistant (to QALead):

**Feature: User Login**

```gherkin
Feature: Login functionality

  Scenario: Successful login with valid credentials
    Given the user is on the login page
    When the user enters a valid username "john_doe" and a valid password "Password123"
    And the user clicks the login button
    Then the user should be redirected to the dashboard

  Scenario: Login fails with a non‑existent username
    Given the user is on the login page
    When the user enters an invalid username "unknown_user" and any password "Password123"
    And the use

ChatResult(chat_id=260199941193231826394198766473087778737, chat_history=[{'content': 'Create 3 BDD test cases for a login feature that includes username and password validation.', 'role': 'assistant', 'name': 'QALead'}, {'content': '**Feature: User Login**\n\n```gherkin\nFeature: Login functionality\n\n  Scenario: Successful login with valid credentials\n    Given the user is on the login page\n    When the user enters a valid username "john_doe" and a valid password "Password123"\n    And the user clicks the login button\n    Then the user should be redirected to the dashboard\n\n  Scenario: Login fails with a non‑existent username\n    Given the user is on the login page\n    When the user enters an invalid username "unknown_user" and any password "Password123"\n    And the user clicks the login button\n    Then the application should display an error message "Username not found"\n\n  Scenario: Login fails when the password is incorrect\n    Given the user is on the login page\n    

# AutoGen BDD Test Case Generator with Ollama

This project creates a multi-agent system where a **QA Lead** instructs a **QA Assistant** to generate BDD (Behavior Driven Development) test cases. The agents communicate using a local LLM hosted via **Ollama**.

## 1. Prerequisites

Before running the code, ensure the following are installed and running:

* **Python**: Version 3.8 or higher.
* **Ollama**: Installed and running locally.
    * **Model**: You must have the specific model pulled.
    * *Command:* `ollama pull gpt-oss:20b-cloud` (or your preferred model).

## 2. Dependencies

Install the required Python packages. We install the OpenAI extension because Ollama is compatible with the OpenAI API format.

```bash
pip install autogen
pip install "autogen[openai]"
```

## 3. Implementation Steps

### Step A: Define LLM Configuration
This is the "brain" of the agent. We must point it to the local Ollama server.

**Crucial Fixes:**
1.  **Base URL**: Added `/v1` to `http://localhost:11434/v1` to match OpenAI standards.
2.  **Model Name**: Removed the `model_name` key (which caused a `TypeError`) and only used `model`.

```python
llm_config = {
    "config_list": [
        {
            "model": "gpt-oss:20b-cloud",  # Must match 'ollama list' exactly
            "base_url": "http://localhost:11434/v1", # required for Ollama compatibility
            "api_key": "ollama", # Placeholder key required by the client
            "temperature": 0.7,
        }
    ]
}
```

### Step B: Create the Assistant Agent (`QAAssistant`)
This agent performs the actual work (writing test cases).

**Crucial Fix:**
* You **must** pass `llm_config` to this agent. If you omit it, the agent acts "brainless" and returns empty strings.

```python
from autogen import AssistantAgent

qa_assistant = AssistantAgent(
    name="QAAssistant",
    description="An assistant that answers questions based on provided context.",
    system_message="You are a helpful QA assistant who can write BDD scenarios.",
    llm_config=llm_config  # <--- WITHOUT THIS, THE AGENT WILL NOT WORK
)
```

### Step C: Create the User Proxy Agent (`QALead`)
This agent acts as the manager who assigns tasks.

* **`code_execution_config=False`**: We disable this because we only want text output (BDD scenarios), not Python code execution.
* **`human_input_mode="NEVER"`**: Allows the script to run automatically to completion without pausing for user input.

```python
from autogen import UserProxyAgent

qa_lead = UserProxyAgent(
    name="QALead",
    description="A QA lead who assigns tasks to the QA assistant.",
    system_message="You are a QA lead who assigns tasks to the QA assistant.",
    code_execution_config=False,
    human_input_mode="NEVER"
)
```

### Step D: Initiate the Chat
The QA Lead starts the workflow by sending a specific prompt to the QA Assistant.

```python
qa_lead.initiate_chat(
    recipient=qa_assistant,
    message="Create 3 BDD test cases for a login feature that includes username and password validation.",
    max_turns=3
)
```

## 4. Troubleshooting Guide

| Issue | Error Message | Cause | Fix |
| :--- | :--- | :--- | :--- |
| **Blank Output** | Agent returns empty replies | `llm_config` was missing in `AssistantAgent`. | Pass `llm_config=llm_config` when creating the agent. |
| **Config Error** | `TypeError: Completions.create() got an unexpected keyword argument 'model_name'` | The config dictionary has keys the API doesn't recognize. | Remove `model_name` from `llm_config`. Use only `model`. |
| **Connection Error** | `Connection refused` | Ollama is not running. | Run `ollama serve` in your terminal. |

## 5. Complete Working Code

```python
from autogen import AssistantAgent, UserProxyAgent

# 1. Configuration (Fixed: Added /v1 and removed model_name)
llm_config = {
    "config_list": [
        {
            "model": "gpt-oss:20b-cloud",
            "base_url": "http://localhost:11434/v1",
            "api_key": "ollama",
            "temperature": 0.7,
        }
    ]
}

# 2. Create Assistant (Fixed: Added llm_config)
qa_assistant = AssistantAgent(
    name="QAAssistant",
    description="An assistant that answers questions based on provided context.",
    system_message="You are a helpful QA assistant who can write BDD scenarios and you create not more than the number of test cases that the user asks for.",
    llm_config=llm_config
)

# 3. Create User Proxy
qa_lead = UserProxyAgent(
    name="QALead",
    description="A QA lead who assigns tasks to the QA assistant.",
    system_message="You are a QA lead who assigns tasks to the QA assistant.",
    code_execution_config=False,
    human_input_mode="NEVER"
)

# 4. Initiate Chat
qa_lead.initiate_chat(
    recipient=qa_assistant,
    message="Create 3 BDD test cases for a login feature that includes username and password validation.",
    max_turns=3
)
```